# MapReduce

### Что такое MapReduce

**MapReduce** — это модель параллельных вычислений и фреймворк для обработки больших объёмов данных (Big Data).
Она была предложена Google в статье *“MapReduce: Simplified Data Processing on Large Clusters”* (2004).
Основная идея — разделить обработку данных на две стадии:

1. **Map (отображение)** — применяет функцию ко всем входным данным и преобразует их в пары (ключ, значение).

2. **Reduce (свёртка)** — агрегирует результаты по ключам, полученным на предыдущем шаге.

![MapReduce](mr.png)

### Скачаем датасет

Создадим рабочую директорию для этого семинара и перейдём в неё. Затем скачаем датасет — это часть архива твитов от российской тролльной фабулы (Internet Research Agency). Файл имеет формат CSV, где каждая строка — отдельный твит с набором полей (автор, текст, язык и т.д.).

In [1]:
! mkdir seminar-2-dir

In [2]:
%cd seminar-2-dir

/home/jovyan/work/seminar2/seminar-2-dir


In [3]:
! curl -o tweets.csv https://raw.githubusercontent.com/fivethirtyeight/russian-troll-tweets/refs/heads/master/IRAhandle_tweets_10.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 90.0M  100 90.0M    0     0  1543k      0  0:00:59  0:00:59 --:--:-- 1914k


Посмотрим на размер файла и его содержимое. `du -h` показывает размер в человекочитаемом виде, `head` выводит первые строки файла — это заголовок с названиями колонок.

In [4]:
! du -h tweets.csv

97M	tweets.csv


In [5]:
! head tweets.csv

external_author_id,author,content,region,language,publish_date,harvested_date,following,followers,updates,post_type,account_type,retweet,account_category,new_june_2018,alt_external_id,tweet_id,article_url,tco1_step1,tco2_step1,tco3_step1
2260338140,POLITICS_T0DAY,https://t.co/9OgJ5RxUEV,United States,Russian,2/16/2016 23:15,2/16/2016 23:16,92,887,12939,,Russian,0,NonEnglish,0,2260338140,699733931055259648,http://twitter.com/politics_t0day/statuses/699733931055259648,https://twitter.com/politics_t0day/status/699733931055259648/photo/1,,
2260338140,POLITICS_T0DAY,Пять этажей жилого дома рухнули в Ярославле после взрыва газа https://t.co/nYwm9SVK9x с помощью @YouTube,United States,Russian,2/16/2016 5:41,2/16/2016 5:42,92,884,12895,,Russian,0,NonEnglish,0,2260338140,699468718888390656,http://twitter.com/politics_t0day/statuses/699468718888390656,https://youtu.be/STxTIceQmsA,,
2260338140,POLITICS_T0DAY,Вербовщика Джихади Джона нашли в Турции через соцсети https://t.co/kJB3G1ioAm с помощью @

Удалим первую строку с заголовком — для MapReduce она не нужна, так как обрабатывалась бы как обычная строка данных. После этого проверим, что заголовок действительно удалён.

In [6]:
! sed -i -e '1d' tweets.csv

In [7]:
! head -n 3 tweets.csv

2260338140,POLITICS_T0DAY,https://t.co/9OgJ5RxUEV,United States,Russian,2/16/2016 23:15,2/16/2016 23:16,92,887,12939,,Russian,0,NonEnglish,0,2260338140,699733931055259648,http://twitter.com/politics_t0day/statuses/699733931055259648,https://twitter.com/politics_t0day/status/699733931055259648/photo/1,,
2260338140,POLITICS_T0DAY,Пять этажей жилого дома рухнули в Ярославле после взрыва газа https://t.co/nYwm9SVK9x с помощью @YouTube,United States,Russian,2/16/2016 5:41,2/16/2016 5:42,92,884,12895,,Russian,0,NonEnglish,0,2260338140,699468718888390656,http://twitter.com/politics_t0day/statuses/699468718888390656,https://youtu.be/STxTIceQmsA,,
2260338140,POLITICS_T0DAY,Вербовщика Джихади Джона нашли в Турции через соцсети https://t.co/kJB3G1ioAm с помощью @YouTube,United States,Russian,2/16/2016 6:10,2/16/2016 6:10,92,884,12896,,Russian,0,NonEnglish,0,2260338140,699476018063659008,http://twitter.com/politics_t0day/statuses/699476018063659008,https://youtu.be/xy5ap3xX_fs,,


### Задача - посчитать количество вхождений каждого слова в текстах сообщений

Попробуем написать наивное решение через Python + класс Counter

Напишем скрипт `simple_counter.py`. Он читает CSV-файл построчно, для каждой строки берёт текст твита (колонка с индексом 2), приводит его к нижнему регистру и с помощью регулярного выражения извлекает слова (последовательности латинских букв). Класс `Counter` накапливает количество вхождений каждого слова. В конце скрипт печатает пары `слово количество`.

In [8]:
%%writefile simple_counter.py

import csv
from collections import Counter
import re

def wordcount_from_csv():
    counter = Counter()
    
    with open('tweets.csv', 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            text = row[2].lower()
            
            words = re.findall(r'[a-z]+', text)
            counter.update(words)
    
    return counter

if __name__ == '__main__':
    c = wordcount_from_csv()
    for key in c:
        print(key, c[key])

Writing simple_counter.py


Запустим скрипт и замерим время выполнения с помощью `%%time`. Результат сохраним в файл `counts.txt`.

In [9]:
%%time
! python3 simple_counter.py > counts.txt

CPU times: user 10.9 ms, sys: 8.7 ms, total: 19.6 ms
Wall time: 1.8 s


Проверим результат: посмотрим на файлы в директории, посчитаем количество уникальных слов (`wc -l`) и выведем первые строки результата.

In [10]:
! ls

counts.txt  simple_counter.py  tweets.csv


In [11]:
! wc -l counts.txt

281837 counts.txt


In [12]:
! head counts.txt

https 172298
t 211293
co 200119
ogj 4
rxuev 1
nywm 1
svk 5
x 3699
youtube 2173
kjb 4


Отсортируем слова по убыванию частоты (`sort -k2,2nr` — числовая сортировка по 2-й колонке в обратном порядке) и выведем топ самых частых слов.

In [13]:
! cat counts.txt | sort -k2,2nr -k1,1 | head 

t 211293
co 200119
https 172298
to 51908
in 44732
s 39609
the 36215
news 34041
of 30243
world 29011
sort: write failed: 'standard output': Broken pipe
sort: write error


### А что если сделать mapreduce через Python и Bash?

**Map** — это первая стадия модели MapReduce, на которой входные данные разбиваются на независимые части и обрабатываются параллельно. Каждая часть поступает в функцию *map*, которая преобразует данные в набор промежуточных пар вида *(ключ, значение)*. Например, при подсчёте слов в тексте каждая строка преобразуется в список пар вроде `("слово", 1)`. Эта стадия отвечает за извлечение и предварительную структуризацию информации.

**Reduce** — это вторая стадия, которая получает сгруппированные по ключу результаты работы map-задач. Функция *reduce* сводит значения, относящиеся к одному ключу, к итоговому результату, например, суммируя, усредняя или объединяя их. В задаче подсчёта слов reduce просто складывает все единицы для каждого слова, чтобы получить общее количество его вхождений. Эта стадия отвечает за агрегацию и формирование конечного вывода.

Интересно, что базовую идею MapReduce можно реализовать даже без Hadoop, используя обычные команды Bash. Поток данных можно пропустить через три этапа: `cat text.txt | map | sort | reduce > result.txt`. Здесь `map` — скрипт, который генерирует пары ключ–значение, `sort` выполняет роль *shuffle and sort*, а `reduce` агрегирует данные по ключам. Такой подход демонстрирует суть парадигмы MapReduce — разделение обработки на этапы отображения, сортировки и свёртки — без необходимости использовать распределённые вычисления.


### Напишем Python скрипт, который сможет выполнять map и reduce

Создадим скрипт `wordcount.py`, который умеет работать в двух режимах:

- **mapper** — читает CSV из `stdin`, для каждой строки извлекает слова из текста (колонка 2) и печатает пары `слово\t1`.
- **reducer** — читает отсортированный поток `слово\tcount` из `stdin`, суммирует счётчики для одинаковых слов и выводит итоговые пары `слово\tсумма`.

Режим передаётся первым аргументом командной строки (`mapper` или `reducer`).

In [14]:
%%writefile wordcount.py
import sys
import csv
import re

def mapper():
    reader = csv.reader(sys.stdin)
    for row in reader:
        text = row[2].lower() 
        words = re.findall(r'[a-z]+', text)
        for word in words:
            print(f"{word}\t1")
        

def reducer():
    current_word = None
    current_count = 0

    for line in sys.stdin:
        word, count = line.strip().split('\t', 1)
        count = int(count)

        if word == current_word or current_word is None:
            current_count += count
        else:
            print(f"{current_word}\t{current_count}")
            current_count = 1
            
        current_word = word
        
    print(f"{current_word}\t{current_count}")

if __name__ == "__main__":
    mode = sys.argv[1]
    if mode == "mapper":
        mapper()
    elif mode == "reducer":
        reducer()


Writing wordcount.py


Соберём весь pipeline в одну bash-команду: `cat` отдаёт файл на вход mapper'у, `sort` выполняет shuffle (группировку по ключу), затем reducer агрегирует данные. Результат сохраним в `counts2.txt` и замерим время.

In [15]:
! ls

counts.txt  simple_counter.py  tweets.csv  wordcount.py


In [16]:
%%time
! cat tweets.csv | python3 wordcount.py mapper | sort -k1,1 | python3 wordcount.py reducer > counts2.txt

CPU times: user 21.1 ms, sys: 12.7 ms, total: 33.8 ms
Wall time: 4.02 s


Проверим результат аналогично первому способу и сравним топ слов с наивным решением.

In [17]:
! ls

counts2.txt  counts.txt  simple_counter.py  tweets.csv	wordcount.py


In [18]:
! cat counts2.txt | sort -k2,2nr -k1,1 | head

t	211293
co	200119
https	172298
to	51908
in	44732
s	39609
the	36215
news	34041
of	30243
world	29011
sort: write failed: 'standard output': Broken pipe
sort: write error


### Напишем теперь аналогичный скрипт для подсчета top10 слов

Скрипт `top10.py` также работает в режимах mapper и reducer. Mapper просто передаёт строки дальше (подготавливая данные для сортировки). Reducer читает отсортированный поток и выводит только первые 10 строк — таким образом мы получаем топ-10 слов. Функция `flush_stdin` дочитывает остаток потока, чтобы избежать ошибокBroken Pipe.

In [19]:
%%writefile top10.py

import sys
from collections import Counter

def flush_stdin():
    for line in sys.stdin:
        pass
        

def mapper():
    for line in sys.stdin:
        print(line.strip() + '\t')

def reducer():
    # читаем поток и выводим только первые 10 строк
    for i, line in enumerate(sys.stdin):
        if i < 10:
            print(line.strip())
        else:
            break
    flush_stdin()

if __name__ == "__main__":
    mode = sys.argv[1]
    if mode == "mapper":
        mapper()
    elif mode == "reducer":
        reducer()


Writing top10.py


Запустим pipeline: отсортируем результаты wordcount по убыванию частоты и отберём топ-10 через `top10.py`. Результат сохраним в `top10.txt`.

In [20]:
%%time
! cat counts2.txt | python3 top10.py mapper | sort -k2,2nr -k1,1 | python3 top10.py reducer > top10.txt

CPU times: user 4.51 ms, sys: 5.17 ms, total: 9.68 ms
Wall time: 490 ms


In [21]:
! cat top10.txt

t	211293
co	200119
https	172298
to	51908
in	44732
s	39609
the	36215
news	34041
of	30243
world	29011


### Так а что с Map Reduce?
С помощью указанных двух скриптов был продемонстрирован основной принцип работы map reduce с помощью уже знакомых bash команд. Давайте теперь перейдем непосредственно к Hadoop MapReduce, ради которого здесь и собрались. Глобально, цель, которую он выполняет, это распределенный запуск скриптов Map & Reduce (на разных воркерах!) и сортировка данных между этими этапами (shuffle).

**Переместим файлы с инпутами в hdfs**

Создадим в HDFS директории для входных данных и загрузим туда файл `tweets.csv`. Команда `hdfs dfs -put` копирует локальный файл в распределённую файловую систему, после чего данные доступны всем воркерам кластера.

In [22]:
! hdfs dfs -mkdir /sem3
! hdfs dfs -mkdir /sem3/wordcount
! hdfs dfs -mkdir /sem3/wordcount/input

2026-09-09 17:43:16,719 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-09-09 17:43:17,914 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-09-09 17:43:18,737 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [23]:
! hdfs dfs -put ./tweets.csv /sem3/wordcount/input/

2026-09-09 17:44:04,687 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [24]:
! hdfs dfs -du -h /sem3/wordcount/input/

2026-09-09 17:44:08,402 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
90.0 M  90.0 M  /sem3/wordcount/input/tweets.csv


**Запустим MapReduce таску**

Запускаем Hadoop Streaming — инструмент, позволяющий использовать произвольные скрипты (в нашем случае Python) в качестве mapper и reducer. Параметры:

- `-files wordcount.py` — отправляет скрипт на все узлы кластера;
- `-mapper` / `-reducer` — команды для запуска map и reduce стадий;
- `-input` / `-output` — пути в HDFS;
- `mapreduce.job.reduces=3` — количество reducer-задач (партий результата).

In [25]:
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.name="word_count" \
-D mapreduce.job.reduces=3 \
-files wordcount.py \
-mapper "python3 wordcount.py mapper" \
-reducer "python3 wordcount.py reducer" \
-input /sem3/wordcount/input/ \
-output /sem3/wordcount/output/

2026-09-09 17:44:54,234 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
packageJobJar: [/tmp/hadoop-unjar12723174830021643751/] [] /tmp/streamjob3457143068329185139.jar tmpDir=null
2026-09-09 17:44:54,562 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.24.0.7:8032
2026-09-09 17:44:54,648 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.24.0.7:8032
2026-09-09 17:44:54,908 INFO  [main] mapreduce.JobResourceUploader (JobResourceUploader.java:disableErasureCodingForPath(907)) - Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1788975064982_0001
2026-09-09 17:44:55,700 INFO  [main] mapred.FileInputFormat (FileInpu

После завершения задачи в директории вывода появляются файлы `part-*` (по одному на каждый reducer) и файл `_SUCCESS` — маркер успешного завершения. Посмотрим содержимое выходной директории.

In [26]:
! hdfs dfs -ls /sem3/wordcount/output

2026-09-09 17:46:19,183 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 4 items
-rw-r--r--   1 hadoop users          0 2026-09-09 17:45 /sem3/wordcount/output/_SUCCESS
-rw-r--r--   1 hadoop users     940563 2026-09-09 17:45 /sem3/wordcount/output/part-00000
-rw-r--r--   1 hadoop users     943203 2026-09-09 17:45 /sem3/wordcount/output/part-00001
-rw-r--r--   1 hadoop users     939845 2026-09-09 17:45 /sem3/wordcount/output/part-00002


Удалим служебный файл `_SUCCESS`, чтобы он не мешал при чтении результатов через `part-*`.

In [27]:
! hdfs dfs -rm /sem3/wordcount/output/_SUCCESS

2026-09-09 17:46:36,006 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Deleted /sem3/wordcount/output/_SUCCESS


Прочитаем результат из HDFS: `part-*` объединяет выходы всех reducer'ов. Затем скачаем результат в локальный файл `result.txt` и выведем топ слов.

In [28]:
! hdfs dfs -cat /sem3/wordcount/output/part-* | head

2026-09-09 17:46:38,888 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
aa	131
aaaaaaaah	1
aaaaand	1
aaaand	1
aaags	1
aaah	1
aaahjkx	1
aaaihta	1
aaailtkoc	1
cat: Unable to write to output stream.
cat: Unable to write to output stream.
cat: Unable to write to output stream.


In [29]:
! hdfs dfs -cat /sem3/wordcount/output/part-* > result.txt

In [30]:
! cat result.txt | sort -k2,2nr -k1,1 | head

t	211293
co	200119
https	172298
to	51908
in	44732
s	39609
the	36215
news	34041
of	30243
world	29011
sort: write failed: 'standard output': Broken pipe
sort: write error


**Давайте теперь запустим top10 скрипт через MapReduce**

Создадим директорию для top10 и удалим старый вывод, если он остался с предыдущего запуска (Hadoop не перезаписывает существующие выходные директории).

In [31]:
! hdfs dfs -mkdir /sem3/top10

2026-09-09 17:47:50,603 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [32]:
! hdfs dfs -rm -r -f /sem3/top10/output

2026-09-09 17:47:55,111 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Запускаем Hadoop Streaming для top10. Здесь появляются новые параметры:

- `KeyFieldBasedComparator` — компаратор для сортировки по полям ключа;
- `stream.num.map.output.key.fields=2` — ключ состоит из двух полей;
- `mapreduce.partition.keycomparator.options='-k2,2nr -k1,1'` — сортировка по 2-му полю (число) по убыванию, затем по 1-му (слово);
- `mapreduce.job.reduces=1` — один reducer, чтобы топ-10 был глобальным.

In [33]:
%%time
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.output.key.comparator.class=org.apache.hadoop.mapreduce.lib.partition.KeyFieldBasedComparator \
-D mapreduce.job.name="top10" \
-D mapreduce.job.reduces=1 \
-D stream.num.map.output.key.fields=2 \
-D mapreduce.partition.keycomparator.options='-k2,2nr -k1,1' \
-files top10.py \
-mapper "python3 top10.py mapper" \
-reducer "python3 top10.py reducer" \
-input /sem3/wordcount/output/ \
-output /sem3/top10/output/

2026-09-09 17:48:01,410 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
packageJobJar: [/tmp/hadoop-unjar8150405334974545536/] [] /tmp/streamjob2144204261799580952.jar tmpDir=null
2026-09-09 17:48:01,665 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.24.0.7:8032
2026-09-09 17:48:01,732 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.24.0.7:8032
2026-09-09 17:48:01,876 INFO  [main] mapreduce.JobResourceUploader (JobResourceUploader.java:disableErasureCodingForPath(907)) - Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1788975064982_0002
2026-09-09 17:48:02,534 INFO  [main] mapred.FileInputFormat (FileInput

Считываем результат — топ-10 самых частых слов в датасете твитов.

In [34]:
! hdfs dfs -cat /sem3/top10/output/part-* 2> /dev/null

2026-09-09 17:48:47,548 WARN  [main] util.NativeCodeLoader (NativeCodeLoader.java:<clinit>(60)) - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
t	211293
co	200119
https	172298
to	51908
in	44732
s	39609
the	36215
news	34041
of	30243
world	29011


### Distributed Cache

Кроме непосредственно **кода** в MapReduce мы также можем передавать на ноды-воркеры еще и другие файлы. Например, мы можем передать файл со словами, которые надо отфильтровать (топ 10 самых популярных слов)

Создадим файл `ban.txt` со списком слов, которые хотим исключить из результатов. Это стоп-слова — самые частые и неинформативные (артикли, предлоги и т.п.).

In [ ]:
%%writefile ban.txt
t
co
https
to
in
s
the
news
of
world

Скрипт `top10_ban.py` отличается от `top10.py` тем, что mapper читает файл `ban.txt` (доступный через Distributed Cache) и фильтрует слова, попавшие в список. Reducer такой же — выводит первые 10 строк отсортированного потока.

In [35]:
%%writefile top10_ban.py

import sys
from collections import Counter

def flush_stdin():
    for line in sys.stdin:
        pass

def get_banned_words():
    with open('ban.txt', 'r') as f:
        return {word.strip() for word in f}

def mapper():
    banned_words = get_banned_words()
    for line in sys.stdin:
        word, count = line.strip().split()
        if word not in banned_words:
            print(line.strip() + '\t')

def reducer():
    # читаем поток и выводим только первые 10 строк
    for i, line in enumerate(sys.stdin):
        if i < 10:
            print(line.strip())
        else:
            break
    flush_stdin()

if __name__ == "__main__":
    mode = sys.argv[1]
    if mode == "mapper":
        mapper()
    elif mode == "reducer":
        reducer()


Writing top10_ban.py


Сначала проверим скрипт локально через bash-pipeline, а затем запустим на Hadoop.

In [ ]:
%%time
! cat counts2.txt | python3 top10_ban.py mapper | sort  -k2,2nr -k1,1 | python3 top10_ban.py reducer > top10_ban.txt

In [ ]:
! cat top10_ban.txt

Запустим на Hadoop. Обратите внимание на параметр `-files top10_ban.py,ban.txt` — теперь на воркеры отправляются сразу два файла: скрипт и файл со списком стоп-слов. Это и есть Distributed Cache.

In [ ]:
! hdfs dfs -mkdir /sem3/top10_ban

In [ ]:
%%time
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.output.key.comparator.class=org.apache.hadoop.mapreduce.lib.partition.KeyFieldBasedComparator \
-D mapreduce.job.name="top10_ban" \
-D mapreduce.job.reduces=1 \
-D stream.num.map.output.key.fields=2 \
-D mapreduce.partition.keycomparator.options='-k2,2nr -k1,1' \
-files top10_ban.py,ban.txt \
-mapper "python3 top10_ban.py mapper" \
-reducer "python3 top10_ban.py reducer" \
-input /sem3/wordcount/output/ \
-output /sem3/top10_ban/output/

In [ ]:
! hdfs dfs -cat /sem3/top10_ban/output/part-*

### Combiner

![Combine](combiner.png)

**Combiner** — это вспомогательный мини-reducer, который выполняется на стороне mapper-узлов и служит для локального агрегирования промежуточных результатов перед отправкой их на этап shuffle. Он позволяет значительно сократить объём передаваемых данных в сеть: например, в задаче WordCount combiner может суммировать количество вхождений слов в пределах одного mapper’а, прежде чем эти частичные суммы будут отправлены на общий reducer. По сути, combiner использует ту же логику, что и reducer, но действует локально и не гарантируется к исполнению Hadoop’ом — это лишь оптимизация, которую фреймворк может применить, если посчитает возможным.


Запустим ту же задачу wordcount, но добавим параметр `-combiner` — ту же функцию reducer'а. Это позволит локально агрегировать данные на стороне mapper, уменьшив объём данных, передаваемых по сети на этапе shuffle. Замерим время и сравним с запуском без combiner.

In [ ]:
! hdfs dfs -mkdir /sem3/wordcount_comb/

In [ ]:
%%time
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.name="word_count" \
-D mapreduce.job.reduces=3 \
-files wordcount.py \
-mapper "python3 wordcount.py mapper" \
-reducer "python3 wordcount.py reducer" \
-combiner "python3 wordcount.py reducer" \
-input /sem3/wordcount/input/ \
-output /sem3/wordcount_comb/output/

In [ ]:
! hdfs dfs -cat /sem3/wordcount_comb/output/part-* | head

### Custom Partitioner

**Partitioner** — это компонент в MapReduce, который определяет, к какому reducer’у будет отправлен каждый ключ после стадии shuffle. Он отвечает за распределение промежуточных пар `key–value` между reduce-задачами, чтобы обеспечить баланс нагрузки и корректную группировку данных: все значения с одинаковым ключом должны попасть на один и тот же reducer. По умолчанию используется `HashPartitioner`, который распределяет ключи по хешу, но при необходимости можно задать **custom partitioner**, если нужно контролировать логику распределения — например, отправлять данные по диапазонам значений, по первым символам ключей или по географическим регионам.


Давайте решим следующую задачу: для каждого пользователя вычислим суммарную длину сообщений, написанных им на каждом языке. В данном случае стандартная сортировка по партициям работать не будет, вопрос - *почему?*

Ключ в этой задаче — составной: `пользователь+язык` (через разделитель `+`). Mapper извлекает автора (колонка 1), текст (колонка 2) и язык (колонка 4), выводит пару `user+lang\tдлина_текста`. Reducer суммирует длины для одинаковых составных ключей.

In [ ]:
%%writefile lang_len.py
import sys
import csv
import re

def mapper():
    reader = csv.reader(sys.stdin)
    for row in reader:
        user, text, lang = row[1], row[2], row[4]
        print(f'{user}+{lang}\t{len(text)}')

def reducer():
    current_word = None
    current_count = 0

    for line in sys.stdin:
        word, count = line.strip().split('\t', 1)
        count = int(count)

        if word == current_word or current_word is None:
            current_count += count
        else:
            print(f"{current_word}\t{current_count}")
            current_count = 1
            
        current_word = word
        
    print(f"{current_word}\t{current_count}")

if __name__ == "__main__":
    mode = sys.argv[1]
    if mode == "mapper":
        mapper()
    elif mode == "reducer":
        reducer()


Проверим скрипт локально через bash-pipeline.

In [ ]:
%%time
! cat tweets.csv | python3 lang_len.py mapper | sort | python3 lang_len.py reducer > lang_len.txt

In [ ]:
! sort -k1,1 lang_len.txt | head

Теперь запустим на Hadoop с custom partitioner. Ключ составной (через `+`), и нам нужно, чтобы партиционирование шло только по первому полю (пользователю), а не по всей строке. Параметры:

- `map.output.key.field.separator='+'` — разделитель полей ключа;
- `KeyFieldBasedPartitioner` — партиционер по заданному полю;
- `mapreduce.partition.keypartitioner.options='-k1,1'` — партиционировать по 1-му полю (пользователю);
- `stream.num.map.output.key.fields=2` — ключ состоит из двух полей;
- `KeyFieldBasedComparator` — сортировка по пользователю, затем по языку.

In [ ]:
! hdfs dfs -mkdir /sem3/lang_len/

In [ ]:
%%time
%%bash
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.name="lang_len" \
-D mapreduce.job.reduces=3 \
-D stream.num.map.output.key.fields=2 \
-D map.output.key.field.separator='+' \
-D mapreduce.partition.keypartitioner.options='-k1,1' \
-D mapreduce.job.output.key.comparator.class=org.apache.hadoop.mapreduce.lib.partition.KeyFieldBasedComparator \
-D mapreduce.partition.keycomparator.options='-k1,1 -k2,2' \
-files lang_len.py \
-mapper "python3 lang_len.py mapper" \
-reducer "python3 lang_len.py reducer" \
-input /sem3/wordcount/input/ \
-output /sem3/lang_len/output/ \
-partitioner org.apache.hadoop.mapred.lib.KeyFieldBasedPartitioner

Посмотрим на результат. Каждый файл `part-*` соответствует одному reducer'у (партиции). Благодаря custom partitioner все записи одного пользователя попали в один и тот же файл, даже если он писал на разных языках.

In [ ]:
! hdfs dfs -ls /sem3/lang_len/output

In [ ]:
! hdfs dfs -cat /sem3/lang_len/output/part-00000 | sort -k1,1 | head -n 20

Проверим корректность партиционирования: найдём все записи пользователя `POLITOPROS` в одной из партиций. Все его записи (для разных языков) должны оказаться в одном файле — это подтверждает, что custom partitioner сработал верно.

In [ ]:
! hdfs dfs -cat /sem3/lang_len/output/part-00002 | grep POLITOPROS